#### 1. Read the fines.csv file that you saved in the previous exercise.

In [1]:

import pandas as pd
import gc

fines = pd.read_csv('../data/fines.csv')
fines

,CarNumber,Make,Model,Refund,Fines,Year
0,Y351O8197RUS,Ford,Focus,1,500.0,2013
1,H917TC36RUS,Ford,Focus,2,8500.0,2000
2,C589EY154RUS,Ford,Focus,1,7500.0,2000
3,K846YE77RUS,Volkswagen,Passat,2,2100.0,1982
4,X4108H125RUS,Ford,Focus,2,1500.0,1994
...,...,...,...,...,...,...
925,Y123XX77,Tesla,Model 3,0,500.0,2022
926,A777AA99,Mercedes-Benz,S-Class,1,2500.0,2021
927,M555MM50,BMW,M5,0,1500.0,2019
928,B001BB11,Audi,RS6,0,3000.0,2023


#### 2. Iterations: in all the following subtasks, you need to calculate fines/refund*year for each row. Create a new column with the calculated data. Measure the time using the magic command %%timeit in the cell.
Write a function that loops through the dataframe using for i in range(0, len(df)), iloc, and append() to a list. Assign the result of the function to a new column in the dataframe.
Do it using iterrows().
Do it using apply() and a lambda function.
Do it using Series objects from the dataframe.
Do it as in the previous subtask, but use the method .values.

In [2]:
%%timeit
def loop_iloc(fines):
    result=[]
    for i in range(0, len(fines)):
        row = fines.iloc[i]
        # if row['Refund'] == 0:
        #     val = 0 
        # else:
        #     val = row['Fines'] / row['Refund'] * row['Year']
        val = row['Fines'] / row['Refund'] * row['Year']
        result.append(val)
    return result

fines['result_iloc']= loop_iloc(fines)
fines

<magic-timeit>:9: RuntimeWarning: divide by zero encountered in scalar divide
<magic-timeit>:9: RuntimeWarning: divide by zero encountered in scalar divide


10.3 ms ± 27.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [3]:
%%timeit
def loop_iterrows(fines):
    result=[]
    for index, row in fines.iterrows():
        if row['Refund'] == 0:
            val = 0 
        else:
            val = row['Fines'] / row['Refund'] * row['Year']
        result.append(val)
    return result

fines['result_iterrows']= loop_iterrows(fines)
fines

10.3 ms ± 155 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
%%timeit
def loop_apply(fines):
    #fines['result_apply'] = fines.apply(lambda row: (row['Fines'] / row['Refund']) * row['Year'], axis=1)
    fines['result_apply'] = fines.apply(lambda row: (row['Fines'] / row['Refund'] * row['Year']) if row['Refund']!=0 else 0 , axis=1)

loop_apply(fines)
fines

3.24 ms ± 66.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit
def loop_series(fines):
    fines['result_series'] = (fines['Fines'] / fines['Refund']) * fines['Year']
    
    #fines['result_series'] = fines['result_series'].where(fines['Refund'] != 0, 0)


loop_series(fines)
fines

54.4 μs ± 6.13 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
%%timeit
def loop_series_values(fines):
    fines['result_series_values'] = fines['Fines'].values / fines['Refund'].values * fines['Year'].values

loop_series_values(fines)
fines

<magic-timeit>:2: RuntimeWarning: divide by zero encountered in divide
<magic-timeit>:2: RuntimeWarning: divide by zero encountered in divide


27.8 μs ± 295 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


#### 3. Indexing: measure the time using the magic command %%timeit in the cell.
Get a row for a specific CarNumber, for example, "O136HO197RUS."
Set the index in your dataframe with CarNumber.
Again, get a row for the same CarNumber. 

In [7]:
%%timeit
fines[fines['CarNumber'] == '9469EX178RUS']


86.8 μs ± 1.91 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
fines.set_index('CarNumber', inplace=True)

In [9]:
%%timeit
fines.loc['9469EX178RUS']

46.2 μs ± 991 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


#### 4.Downcasting:
Run df.info(memory_usage='deep'), and pay attention to the Dtype and memory usage.
Make a copy() of your initial dataframe into another dataframe, optimized_df.
Downcast from float64 to float32 for all columns.
Downcast from int64 to the smallest numerical Dtype possible.
Run info(memory_usage='deep') for your new dataframe. Pay attention to the Dtype and memory usage.

In [10]:
fines.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y351O8197RUS to E999EE178
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Make                  930 non-null    object 
 1   Model                 919 non-null    object 
 2   Refund                930 non-null    int64  
 3   Fines                 930 non-null    float64
 4   Year                  930 non-null    int64  
 5   result_iloc           930 non-null    float64
 6   result_iterrows       930 non-null    float64
 7   result_apply          930 non-null    float64
 8   result_series         930 non-null    float64
 9   result_series_values  930 non-null    float64
dtypes: float64(6), int64(2), object(2)
memory usage: 243.4 KB


In [11]:
optimized_df = fines.copy()

In [12]:

float_cols = optimized_df.select_dtypes(include=['float64']).columns
optimized_df[float_cols] = optimized_df[float_cols].astype('float32')

int_cols = optimized_df.select_dtypes(include=['int64']).columns
for col in int_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='integer')

optimized_df.info(memory_usage='deep')



<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y351O8197RUS to E999EE178
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Make                  930 non-null    object 
 1   Model                 919 non-null    object 
 2   Refund                930 non-null    int8   
 3   Fines                 930 non-null    float32
 4   Year                  930 non-null    int16  
 5   result_iloc           930 non-null    float32
 6   result_iterrows       930 non-null    float32
 7   result_apply          930 non-null    float32
 8   result_series         930 non-null    float32
 9   result_series_values  930 non-null    float32
dtypes: float32(6), int16(1), int8(1), object(2)
memory usage: 209.8 KB


#### 5. Categories:
Change the object type columns to category.
This time, check the memory usage. It will probably decrease by 2–3 times compared to the initial dataframe.

In [13]:
object_cols = optimized_df.select_dtypes(include=['object']).columns
optimized_df[object_cols] = optimized_df[object_cols].astype('category')
optimized_df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y351O8197RUS to E999EE178
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Make                  930 non-null    category
 1   Model                 919 non-null    category
 2   Refund                930 non-null    int8    
 3   Fines                 930 non-null    float32 
 4   Year                  930 non-null    int16   
 5   result_iloc           930 non-null    float32 
 6   result_iterrows       930 non-null    float32 
 7   result_apply          930 non-null    float32 
 8   result_series         930 non-null    float32 
 9   result_series_values  930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 116.0 KB


#### 6. Memory clean:
Using the library gc and the command %reset_selective, clean the memory of your initial dataframe only.

In [14]:
%reset_selective -f ^fines$
gc.collect()

64